# grlc

grlc treats each SPARQL query as a first-class artefact stored in a Git repository: every query file becomes one operation, and the API version is the commit. It reaches both endpoints, but with one operation each, so it cannot combine them.

In [1]:
from helper import call

## The join

One operation per endpoint, no join.

In [2]:
call("http://localhost:8082/api-local/article-meta?doi=10.1007/s11192-022-04367-w", headers={"Accept": "application/json"})

curl -H 'Accept: application/json' 'http://localhost:8082/api-local/article-meta?doi=10.1007/s11192-022-04367-w' 

# 200 OK

{
  "head": {
    "link": [],
    "vars": [
      "title",
      "br"
    ]
  },
  "results": {
    "distinct": false,
    "ordered": true,
    "bindings": [
      {
        "title": {
          "type": "literal",
          "value": "Identifying And Correcting Invalid Citations Due To DOI Errors In Crossref Data"
        },
        "br": {
          "type": "uri",
          "value": "https://w3id.org/oc/meta/br/061202127149"
        }
      }
    ]
  }
}


In [3]:
call("http://localhost:8082/api-local/citations-index?br=https://w3id.org/oc/meta/br/061202127149", headers={"Accept": "application/json"})

curl -H 'Accept: application/json' 'http://localhost:8082/api-local/citations-index?br=https://w3id.org/oc/meta/br/061202127149' 



# 200 OK

{
  "head": {
    "vars": [
      "br",
      "oc_citation_count"
    ]
  },
  "results": {
    "bindings": [
      {
        "br": {
          "type": "uri",
          "value": "https://w3id.org/oc/meta/br/061202127149"
        },
        "oc_citation_count": {
          "datatype": "http://www.w3.org/2001/XMLSchema#int",
          "type": "literal",
          "value": "6"
        }
      }
    ]
  },
  "meta": {
    "query-time-ms": 40,
    "result-size-total": 1
  }
}


## Output

Format follows the `Accept` header, forwarded to the endpoint.

In [4]:
call("http://localhost:8082/api-local/article-meta?doi=10.1007/s11192-022-04367-w", headers={"Accept": "text/csv"})

curl -H 'Accept: text/csv' 'http://localhost:8082/api-local/article-meta?doi=10.1007/s11192-022-04367-w' 



# 200 OK

"title","br"
"Identifying And Correcting Invalid Citations Due To DOI Errors In Crossref Data","https://w3id.org/oc/meta/br/061202127149"


## Pagination

The `last` link is a guess: grlc does not get the real total because of a bug ([CLARIAH/grlc#573](https://github.com/CLARIAH/grlc/issues/573)).

In [5]:
call("http://localhost:8082/api-local/citing-list?br=https://w3id.org/oc/meta/br/061202127149&page=1", headers={"Accept": "application/json"}, show_headers=True)

curl -i -H 'Accept: application/json' 'http://localhost:8082/api-local/citing-list?br=https://w3id.org/oc/meta/br/061202127149&page=1' 



# 200 OK
# Content-Type: application/sparql-results+json
# Link: <http://localhost:8082/api-local/citing-list?br=https%3A%2F%2Fw3id.org%2Foc%2Fmeta%2Fbr%2F061202127149&page=2>; rel=next, <http://localhost:8082/api-local/citing-list?br=https%3A%2F%2Fw3id.org%2Foc%2Fmeta%2Fbr%2F061202127149&page=0>; rel=prev, <http://localhost:8082/api-local/citing-list?br=https%3A%2F%2Fw3id.org%2Foc%2Fmeta%2Fbr%2F061202127149&page=1>; rel=first, <http://localhost:8082/api-local/citing-list?br=https%3A%2F%2Fw3id.org%2Foc%2Fmeta%2Fbr%2F061202127149&page=500.0>; rel=last

[
  {
    "citing": "https://w3id.org/oc/meta/br/0609403935"
  },
  {
    "citing": "https://w3id.org/oc/meta/br/0609604964"
  }
]


## Versioning

The version is the Git commit. With a `GITHUB_TOKEN` in `.env` and the queries pushed to GitHub, grlc serves any past revision through the `api-git` route, naming the commit in the path:

```sh
curl -H 'Accept: application/json' \
  'http://localhost:8082/api-git/arcangelo7/sparql-api-comparison/subdir/grlc/queries/commit/1872cd59b0524ce764060ba72f61d5e7aafe29a4/article-meta?doi=10.1007/s11192-022-04367-w'
```

## API description

OpenAPI 2.0.

In [6]:
call("http://localhost:8082/api-local/swagger")

curl http://localhost:8082/api-local/swagger 

# 200 OK

{
  "basePath": "/api-local/",
  "definitions": {
    "Message": {
      "type": "string"
    }
  },
  "host": "localhost",
  "info": {
    "contact": {
      "name": "",
      "url": ""
    },
    "description": "API generated from local files",
    "title": "local",
    "version": "local"
  },
  "next_commit": null,
  "paths": {
    "/article-meta": {
      "get": {
        "description": "\n\n```\n#+ summary: Title of a bibliographic resource in OpenCitations Meta, by DOI\n#+ endpoint: https://sparql.opencitations.net/meta\n#+ method: GET\n#+ tags:\n#+   - meta\n\nPREFIX datacite: <http://purl.org/spar/datacite/>\nPREFIX literal: <http://www.essepuntato.it/2010/06/literalreification/>\nPREFIX dcterms: <http://purl.org/dc/terms/>\n\nSELECT ?title ?br WHERE {\n  ?id datacite:usesIdentifierScheme datacite:doi ;\n      literal:hasLiteralValue ?_doi .\n  ?br datacite:hasIdentifier ?id ;\n      dcterms:title ?title .\n}\n\n```",
   

## Authentication

Not supported.